In [5]:
# Part 1: Python Foundations & Data Structures
import pandas as pd
from datetime import datetime

# 1. Setup Configuration
CONFIG = {
    "countries": ["US", "GB", "IN"],
    "data_path": "data/",
    "max_rows_per_file": 20000
}

print(f"=== TrendPulse Part 1 Execution Started ===")
print(f"Target Countries: {CONFIG['countries']}")
print(f"Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")

# 2. Loading Function
def load_country(path, country, max_rows=None):
    file_path = f"{path}{country}videos.csv"
    df = pd.read_csv(file_path, nrows=max_rows)
    df['country'] = country
    records = df.to_dict("records")
    print(f"Loaded {len(records)} records for {country}.")
    return records

# Load category mapping dictionary
cat_df = pd.read_csv(f"{CONFIG['data_path']}categories.csv")
category_names = dict(zip(cat_df['category_id'], cat_df['category_name']))

# 3. Type Conversion and Cleaning
raw_records = []
for country in CONFIG["countries"]:
    country_records = load_country(CONFIG["data_path"], country, CONFIG["max_rows_per_file"])
    for rec in country_records:
        try:
            rec['views'] = int(rec['views']) if pd.notna(rec['views']) else 0
            rec['likes'] = int(rec['likes']) if pd.notna(rec['likes']) else 0
            rec['dislikes'] = int(rec['dislikes']) if pd.notna(rec['dislikes']) else 0
            rec['comment_count'] = int(rec['comment_count']) if pd.notna(rec['comment_count']) else 0
            rec['comments_disabled'] = bool(rec['comments_disabled'])
            rec['ratings_disabled'] = bool(rec['ratings_disabled'])
        except (ValueError, TypeError):
            # Edge case handling for malformed data
            rec['views'], rec['likes'], rec['dislikes'], rec['comment_count'] = 0, 0, 0, 0
        raw_records.append(rec)

print(f"Total cleaned records loaded: {len(raw_records)}")

# 4. Core Data Structures (Tuples & Dictionaries)
unique_video_ids = set((r['video_id'], r['country']) for r in raw_records)
print(f"Unique video ID tuples count: {len(unique_video_ids)}")

channel_counts = {}
for r in raw_records:
    ch = r['channel_title']
    channel_counts[ch] = channel_counts.get(ch, 0) + 1

top_15_channels = sorted(channel_counts.items(), key=lambda x: x[1], reverse=True)[:15]

# 5. Variable Scope Demonstration
global_scope_var = "I am accessible anywhere"
def scope_demonstration():
    local_scope_var = "I am only inside this function"
    return local_scope_var

# 6. Loops over Strings & Sequences
def title_features(title):
    if not isinstance(title, str):
        return {"upper_words": 0, "exclamations": 0, "word_count": 0}
    words = title.split()
    upper_words = sum(1 for w in words if w.isupper() and len(w) > 1)
    exclamations = title.count('!')
    return {"upper_words": upper_words, "exclamations": exclamations, "word_count": len(words)}

# 7. Rule-Based Classification
def classify_video(record):
    views = record.get('views', 0)
    likes = record.get('likes', 0)
    dislikes = record.get('dislikes', 0)

    if views == 0:
        return "Unrated"

    like_ratio = likes / (likes + dislikes) if (likes + dislikes) > 0 else 0

    if views > 1000000 and like_ratio > 0.95:
        return "Viral Hit"
    elif like_ratio > 0.90:
        return "Steady Performer"
    elif like_ratio < 0.60:
        return "Divisive"
    else:
        return "Low Engagement"

classification_counts = {}
for r in raw_records:
    label = classify_video(r)
    classification_counts[label] = classification_counts.get(label, 0) + 1

print("Classification Distribution:", classification_counts)

=== TrendPulse Part 1 Execution Started ===
Target Countries: ['US', 'GB', 'IN']
Timestamp: 2026-11-07 06:48:32

Loaded 16080 records for US.
Loaded 15075 records for GB.
Loaded 15075 records for IN.
Total cleaned records loaded: 46230
Unique video ID tuples count: 15059
Classification Distribution: {'Steady Performer': 33859, 'Low Engagement': 6093, 'Divisive': 807, 'Viral Hit': 5471}
